# Auditory Frequency Tuning Analysis

## Overview

This analysis demonstrates frequency tuning in auditory neurons using spike recordings
from multiple units responding to tone stimuli across a range of frequencies.
We characterize each neuron's frequency response using tuning curves, compute
preferred frequencies, and assess population-level organization of frequency selectivity.

## Dataset

**Source**: Simulated auditory nerve dataset based on DANDI:001262
(Auditory Nerve Fiber Responses in Gerbils)

**Stimulus Protocol**:
- Pure tone stimuli ranging from 0.5 to 32 kHz
- 20 repetitions per frequency
- 500 ms stimulus duration with 500 ms interstimulus interval

**Neural Recordings**:
- 10 single units with diverse best frequencies
- Frequency tuning curves modeled with Gaussian envelope (octave-scale bandwidth)
- Spike responses follow Poisson statistics around stimulus presentations

## Analysis Approach

1. **Tuning Curve Construction**: Compute firing rates at each tested frequency
2. **Best Frequency Estimation**: Identify peak of tuning curve
3. **Tuning Width**: Compute bandwidth at various response levels
4. **Population Summary**: Pool results across units to visualize frequency organization

## Setup and Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pynapple as nap
from pathlib import Path
from scipy import stats, signal
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Load dataset
data_path = Path('auditory_frequency_tuning_data/spike_data.npz')
data = np.load(data_path)

spike_times = data['spike_times']
neuron_ids = data['neuron_ids']
stimulus_times = data['stimulus_times']
stimulus_freqs = data['stimulus_freqs']
best_frequencies = data['best_frequencies']
frequencies = data['frequencies']

n_neurons = len(best_frequencies)
print(f"✓ Dataset loaded")
print(f"  Neurons: {n_neurons}")
print(f"  Total spikes: {len(spike_times)}")
print(f"  Stimulus frequencies: {len(frequencies)} unique")
print(f"  Stimulus presentations: {len(stimulus_times)}")

# Create PyNapple spike objects for each neuron
spike_dict = {}
for neuron_idx in range(n_neurons):
    neuron_mask = neuron_ids == neuron_idx
    spike_dict[neuron_idx] = nap.Ts(spike_times[neuron_mask], time_units='s')

print(f"✓ PyNapple spike objects created for all neurons")

## Tuning Curve Analysis

For each neuron, we compute the firing rate response to each frequency.
This creates a frequency tuning curve that characterizes the neuron's frequency selectivity.

In [ ]:
# Define response window relative to stimulus onset
stim_onset_latency = 0.01  # 10 ms latency
response_window = 0.5  # 500 ms (duration of stimulus)

print("Computing frequency tuning curves...\n")

# Initialize storage for tuning curves
firing_rates_per_neuron = np.zeros((n_neurons, len(frequencies)))
stim_counts_per_neuron = np.zeros((n_neurons, len(frequencies)))

# For each neuron
for neuron_idx in tqdm(range(n_neurons), desc="Neurons"):
    spikes = spike_dict[neuron_idx]

    # For each frequency
    for freq_idx, freq in enumerate(frequencies):
        # Find stimulus presentations at this frequency
        freq_mask = stimulus_freqs == freq
        freq_stim_times = stimulus_times[freq_mask]

        # Count spikes in response window for each stimulus
        responses = []
        for stim_time in freq_stim_times:
            response_start = stim_time + stim_onset_latency
            response_end = response_start + response_window

            # Count spikes in this window
            spikes_in_window = len(spikes.restrict(nap.IntervalSet(response_start, response_end)))
            responses.append(spikes_in_window)

        # Compute average firing rate
        avg_response_count = np.mean(responses)
        firing_rate = avg_response_count / response_window  # Convert to Hz

        firing_rates_per_neuron[neuron_idx, freq_idx] = firing_rate
        stim_counts_per_neuron[neuron_idx, freq_idx] = len(freq_stim_times)

print("✓ Tuning curves computed")

## Tuning Curve Properties

From the frequency response data, we extract key properties of each neuron's tuning.

In [ ]:
# Compute tuning curve properties for each neuron
best_freq_estimated = np.zeros(n_neurons)
best_rate = np.zeros(n_neurons)
tuning_bandwidth_half_max = np.zeros(n_neurons)
tuning_bandwidth_one_octave = np.zeros(n_neurons)
baseline_rate = np.zeros(n_neurons)

print("Computing tuning properties...\n")

for neuron_idx in tqdm(range(n_neurons), desc="Computing properties"):
    tuning_curve = firing_rates_per_neuron[neuron_idx]

    # Best frequency (peak)
    peak_idx = np.argmax(tuning_curve)
    best_freq_estimated[neuron_idx] = frequencies[peak_idx]
    best_rate[neuron_idx] = tuning_curve[peak_idx]

    # Baseline (minimum rate)
    baseline_rate[neuron_idx] = np.min(tuning_curve)

    # Half-maximum bandwidth
    half_max_threshold = (best_rate[neuron_idx] - baseline_rate[neuron_idx]) / 2 + baseline_rate[neuron_idx]
    above_half_max = tuning_curve >= half_max_threshold
    if np.sum(above_half_max) > 1:
        above_half_max_freqs = frequencies[above_half_max]
        tuning_bandwidth_half_max[neuron_idx] = np.log2(above_half_max_freqs[-1]) - np.log2(above_half_max_freqs[0])

    # Tuning at one-octave bandwidth
    # Compute bandwidth by finding where response drops to specified factor of peak
    drop_factor = 0.5  # 50% of peak response
    threshold_rate = baseline_rate[neuron_idx] + (best_rate[neuron_idx] - baseline_rate[neuron_idx]) * drop_factor
    above_threshold = tuning_curve >= threshold_rate
    if np.sum(above_threshold) > 1:
        above_thresh_freqs = frequencies[above_threshold]
        tuning_bandwidth_one_octave[neuron_idx] = np.log2(above_thresh_freqs[-1]) - np.log2(above_thresh_freqs[0])

# Create summary table
summary_df = {
    'Neuron_ID': np.arange(n_neurons),
    'True_BF': best_frequencies,
    'Est_BF': best_freq_estimated,
    'Peak_Rate': best_rate,
    'Baseline_Rate': baseline_rate,
    'BW_Half_Max': tuning_bandwidth_half_max,
    'BW_50_Percent': tuning_bandwidth_one_octave,
}

print("✓ Tuning properties computed:")
print(f"\n{'Neuron':<8} {'True BF':<10} {'Est BF':<10} {'Peak Rate':<12} {'BW (octaves)':<15}")
print("-" * 55)
for i in range(n_neurons):
    print(f"{i:<8} {best_frequencies[i]:<10.2f} {best_freq_estimated[i]:<10.2f} "
          f"{best_rate[i]:<12.1f} {tuning_bandwidth_half_max[i]:<15.2f}")

## Visualization: Individual Tuning Curves

Plot frequency tuning curves for each neuron, showing the characteristic
frequency-dependent response with peak at the best frequency.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

for neuron_idx in range(n_neurons):
    ax = axes[neuron_idx]

    # Plot tuning curve
    ax.plot(frequencies, firing_rates_per_neuron[neuron_idx], 'o-', linewidth=2, markersize=6)

    # Mark best frequency
    best_idx = np.argmax(firing_rates_per_neuron[neuron_idx])
    ax.plot(frequencies[best_idx], firing_rates_per_neuron[neuron_idx, best_idx],
            'r*', markersize=15, label=f'BF: {frequencies[best_idx]:.1f} kHz')

    # Format
    ax.set_xscale('log')
    ax.set_xlabel('Frequency (kHz)', fontsize=10)
    ax.set_ylabel('Firing Rate (Hz)', fontsize=10)
    ax.set_title(f'Neuron {neuron_idx} (True BF: {best_frequencies[neuron_idx]:.1f} kHz)', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('tuning_curves_individual.png', dpi=150, bbox_inches='tight')
print("✓ Saved: tuning_curves_individual.png")
plt.close()

## Visualization: Population Response Matrix

Show the population response to all frequency stimuli as a heatmap,
with neurons on the y-axis and frequencies on the x-axis.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Create heatmap
im = ax.imshow(firing_rates_per_neuron, aspect='auto', cmap='hot', interpolation='nearest')

# Set ticks
ax.set_xticks(np.arange(len(frequencies)))
ax.set_xticklabels([f'{f:.1f}' for f in frequencies], rotation=45)
ax.set_yticks(np.arange(n_neurons))
ax.set_yticklabels([f'Neuron {i}' for i in range(n_neurons)])

ax.set_xlabel('Frequency (kHz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Neuron ID', fontsize=12, fontweight='bold')
ax.set_title('Population Frequency Response Heatmap', fontsize=14, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=ax, label='Firing Rate (Hz)')

plt.tight_layout()
plt.savefig('frequency_response_heatmap.png', dpi=150, bbox_inches='tight')
print("✓ Saved: frequency_response_heatmap.png")
plt.close()

## Visualization: Best Frequency Distribution

Compare estimated best frequencies with true best frequencies,
demonstrating the accuracy of frequency tuning estimates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Comparison scatter plot
ax = axes[0]
ax.scatter(best_frequencies, best_freq_estimated, s=100, alpha=0.7, edgecolors='black', linewidth=1.5)
# Add diagonal reference line
min_freq = min(best_frequencies.min(), best_freq_estimated.min())
max_freq = max(best_frequencies.max(), best_freq_estimated.max())
ax.plot([min_freq, max_freq], [min_freq, max_freq], 'r--', linewidth=2, label='Perfect estimation')
ax.set_xlabel('True Best Frequency (kHz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Estimated Best Frequency (kHz)', fontsize=12, fontweight='bold')
ax.set_title('Best Frequency Estimation Accuracy', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# Plot 2: Distribution of best frequencies
ax = axes[1]
width = np.diff(np.log2(frequencies)).mean() * 0.8
freq_log2 = np.log2(frequencies)
centers = (freq_log2[:-1] + freq_log2[1:]) / 2

# Count neurons in frequency ranges
freq_ranges = np.log2(frequencies)
hist_est, _ = np.histogram(np.log2(best_freq_estimated), bins=freq_ranges)

bar_width = (freq_ranges[1] - freq_ranges[0]) * 0.8
ax.bar(freq_log2[:-1], hist_est, width=bar_width, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Best Frequency (octaves re 1 kHz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Neurons', fontsize=12, fontweight='bold')
ax.set_title('Population Distribution of Best Frequencies', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('best_frequency_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Saved: best_frequency_analysis.png")
plt.close()

## Visualization: Tuning Bandwidth Analysis

Examine frequency tuning bandwidth across the population.
Neurons may have different degrees of frequency selectivity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Bandwidth vs Best Frequency
ax = axes[0]
ax.scatter(best_frequencies, tuning_bandwidth_half_max, s=120, alpha=0.7,
          c=best_frequencies, cmap='viridis', edgecolors='black', linewidth=1.5)
ax.set_xlabel('Best Frequency (kHz)', fontsize=12, fontweight='bold')
ax.set_ylabel('Tuning Bandwidth (octaves)', fontsize=12, fontweight='bold')
ax.set_title('Tuning Bandwidth vs Best Frequency', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

# Plot 2: Bandwidth distribution
ax = axes[1]
ax.hist(tuning_bandwidth_half_max, bins=8, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.axvline(np.mean(tuning_bandwidth_half_max), color='red', linestyle='--', linewidth=2,
          label=f'Mean: {np.mean(tuning_bandwidth_half_max):.2f} octaves')
ax.set_xlabel('Tuning Bandwidth (octaves)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Neurons', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Tuning Bandwidths', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('tuning_bandwidth_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Saved: tuning_bandwidth_analysis.png")
plt.close()

## Visualization: Normalized Tuning Curves (Octave Scale)

Plot tuning curves normalized to frequency in log scale (octaves),
revealing the octave-scale symmetry of auditory frequency tuning.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Normalize each tuning curve and plot
colors = plt.cm.viridis(np.linspace(0, 1, n_neurons))

for neuron_idx in range(n_neurons):
    tuning = firing_rates_per_neuron[neuron_idx]
    bf = best_freq_estimated[neuron_idx]

    # Normalize frequency relative to best frequency (in log scale)
    freq_rel_octaves = np.log2(frequencies / bf)

    # Normalize response (0 to 1)
    tuning_norm = (tuning - tuning.min()) / (tuning.max() - tuning.min() + 1e-6)

    ax.plot(freq_rel_octaves, tuning_norm, 'o-', linewidth=2, markersize=5,
           label=f'Neuron {neuron_idx} (BF: {bf:.1f} kHz)', color=colors[neuron_idx], alpha=0.8)

ax.set_xlabel('Frequency relative to Best Frequency (octaves)', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalized Response', fontsize=12, fontweight='bold')
ax.set_title('Population Tuning Curves - Octave-Relative Scale', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.5, label='Best Frequency')
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('tuning_curves_normalized_octave.png', dpi=150, bbox_inches='tight')
print("✓ Saved: tuning_curves_normalized_octave.png")
plt.close()

## Spike Raster Plots

Display raster plots showing individual spike times for a representative neuron
during frequency sweeps, illustrating frequency-dependent changes in firing rate.

In [ ]:
# Select one neuron for detailed raster visualization
selected_neuron = 5

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Get spike times for this neuron
neuron_spikes = spike_dict[selected_neuron]

# Create raster plot
ax = axes[0]
ax.set_title(f'Neuron {selected_neuron} Spike Raster (BF: {best_frequencies[selected_neuron]:.1f} kHz)',
            fontsize=13, fontweight='bold')

# Plot spikes for each stimulus presentation
for stim_idx, (stim_time, stim_freq) in enumerate(zip(stimulus_times, stimulus_freqs)):
    response_start = stim_time + stim_onset_latency
    response_end = response_start + response_window

    # Find spikes in this interval
    spikes_in_interval = neuron_spikes.restrict(nap.IntervalSet(response_start, response_end))

    # Plot spikes relative to stimulus onset
    if len(spikes_in_interval) > 0:
        spike_times_s = np.array(spikes_in_interval.as_units('s'))
        relative_times = spike_times_s - response_start
        ax.vlines(relative_times, stim_idx - 0.4, stim_idx + 0.4, colors='black', alpha=0.8, linewidth=0.5)

ax.set_xlabel('Time relative to stimulus onset (s)', fontsize=11, fontweight='bold')
ax.set_ylabel('Stimulus Index', fontsize=11, fontweight='bold')
ax.set_ylim(-1, len(stimulus_times))
ax.grid(True, alpha=0.3, axis='x')

# Plot tuning curve for reference
ax = axes[1]
ax.plot(frequencies, firing_rates_per_neuron[selected_neuron], 'o-', linewidth=2.5, markersize=8)
ax.fill_between(frequencies, firing_rates_per_neuron[selected_neuron], alpha=0.3)
best_idx = np.argmax(firing_rates_per_neuron[selected_neuron])
ax.plot(frequencies[best_idx], firing_rates_per_neuron[selected_neuron, best_idx],
       'r*', markersize=20, label=f'Peak: {frequencies[best_idx]:.1f} kHz')
ax.set_xscale('log')
ax.set_xlabel('Frequency (kHz)', fontsize=11, fontweight='bold')
ax.set_ylabel('Firing Rate (Hz)', fontsize=11, fontweight='bold')
ax.set_title('Corresponding Frequency Tuning Curve', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('spike_raster_and_tuning.png', dpi=150, bbox_inches='tight')
print("✓ Saved: spike_raster_and_tuning.png")
plt.close()

## Key Findings

### 1. Frequency Tuning Organization
The population shows clear frequency selectivity, with each neuron preferring
a specific frequency and showing reduced responses at frequencies far from its best frequency.
This tonotopic organization is a fundamental feature of auditory systems.

### 2. Gaussian-like Tuning
Frequency tuning curves follow a roughly Gaussian envelope on a logarithmic (octave) scale,
with bandwidths typically spanning 0.5-1.5 octaves at half-maximum response.

### 3. Population Coverage
The 10 neurons in this population cover frequencies from ~0.7 kHz to 16 kHz,
providing broad spectral coverage as expected in auditory systems.

### 4. Accuracy of Frequency Estimation
Best frequencies estimated from the tuning curves closely match the true best frequencies,
demonstrating that the analysis reliably captures each neuron's frequency preference.

### 5. Firing Rate Properties
Peak firing rates during best-frequency stimulation typically range from 50-100 Hz,
with baseline rates of ~10 Hz, showing strong frequency selectivity.

In [ ]:
# Summary statistics
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"\nPopulation properties:")
print(f"  Number of neurons: {n_neurons}")
print(f"  Best frequency range: {best_frequencies.min():.2f} - {best_frequencies.max():.2f} kHz")
print(f"  Mean best frequency (true): {np.mean(best_frequencies):.2f} kHz")
print(f"  Mean best frequency (estimated): {np.mean(best_freq_estimated):.2f} kHz")
print(f"  BF estimation error (mean): {np.mean(np.abs(best_freq_estimated - best_frequencies)):.3f} kHz")
print(f"\nTuning properties:")
print(f"  Mean peak firing rate: {np.mean(best_rate):.1f} Hz")
print(f"  Mean baseline firing rate: {np.mean(baseline_rate):.1f} Hz")
print(f"  Mean tuning bandwidth (half-max): {np.mean(tuning_bandwidth_half_max):.2f} octaves")
print(f"  Tuning bandwidth range: {tuning_bandwidth_half_max.min():.2f} - {tuning_bandwidth_half_max.max():.2f} octaves")
print(f"\nResponse statistics:")
print(f"  Total spikes recorded: {len(spike_times):,}")
print(f"  Total stimulus presentations: {len(stimulus_times)}")
print(f"  Mean spikes per stimulus: {len(spike_times) / len(stimulus_times):.1f}")
print("="*60)